In [ ]:
from google.colab import files
uploaded = files.upload()

Saving student_performance_interactions (1).csv to student_performance_interactions (1).csv


In [ ]:
import pandas as pd
import time
import warnings

# ✅ Ignore warnings (including your datetime warning)
warnings.filterwarnings("ignore")

from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("student_performance_interactions (1).csv")

if "student_id" in df.columns:
    df = df.drop("student_id", axis=1)

# =========================
# 2. DISCRETIZATION
# =========================
df['study_level'] = pd.cut(df['daily_study_hours'],
                          bins=[0,2,5,10],
                          labels=['Low','Medium','High'])

df['sleep_level'] = pd.cut(df['sleep_hours'],
                          bins=[0,5,7,10],
                          labels=['Low','Normal','High'])

df['attendance_level'] = pd.cut(df['attendance_percentage'],
                               bins=[0,60,80,100],
                               labels=['Low','Medium','High'])

df['anxiety_level'] = pd.cut(df['exam_anxiety_score'],
                            bins=[0,3,6,10],
                            labels=['Low','Medium','High'])

# =========================
# 3. SELECT DATA
# =========================
df_ar = df[['study_level','sleep_level','attendance_level',
            'anxiety_level','grade']].astype(str)

# =========================
# 4. ENCODING
# =========================
df_encoded = pd.get_dummies(df_ar)

# =========================
# 5. APRIORI
# =========================
start = time.time()

apriori_itemsets = apriori(df_encoded,
                          min_support=0.1,
                          use_colnames=True)

apriori_rules = association_rules(apriori_itemsets,
                                 metric="confidence",
                                 min_threshold=0.6)

apriori_time = time.time() - start

# =========================
# 6. FP-GROWTH
# =========================
start = time.time()

fp_itemsets = fpgrowth(df_encoded,
                      min_support=0.1,
                      use_colnames=True)

fp_rules = association_rules(fp_itemsets,
                            metric="confidence",
                            min_threshold=0.6)

fp_time = time.time() - start

# =========================
# 7. FILTER STRONG RULES
# =========================
def filter_rules(rules):
    # Balanced filtering
    rules = rules[(rules['confidence'] > 0.6) & (rules['lift'] > 1.0)]

    # Keep rules involving grade (either side)
    rules = rules[
        rules['antecedents'].astype(str).str.contains('grade') |
        rules['consequents'].astype(str).str.contains('grade')
    ]

    # Sort
    rules = rules.sort_values(by='lift', ascending=False)

    return rules

apriori_rules = filter_rules(apriori_rules)
fp_rules = filter_rules(fp_rules)

# =========================
# 8. DISPLAY RESULTS
# =========================
print("\n🔹 APRIORI (Filtered Rules):\n")
print(apriori_rules[['antecedents','consequents','support','confidence','lift']].head(5))

print("\n🔹 FP-GROWTH (Filtered Rules):\n")
print(fp_rules[['antecedents','consequents','support','confidence','lift']].head(5))

# =========================
# 9. COMPARISON
# =========================
print("\n📊 PERFORMANCE COMPARISON:\n")
print(f"Apriori Time: {apriori_time:.4f} sec")
print(f"FP-Growth Time: {fp_time:.4f} sec")

print(f"Apriori Rules: {len(apriori_rules)}")
print(f"FP-Growth Rules: {len(fp_rules)}")

# =========================
# 10. SAMPLE INPUT
# =========================
sample = {
    "study_level": "Medium",
    "sleep_level": "Normal",
    "attendance_level": "Medium",
    "anxiety_level": "Medium"
}

print("\n🎯 Sample Input:", sample)

sample_set = set([f"{k}_{v}" for k,v in sample.items()])

# =========================
# 11. MATCH RULES
# =========================
print("\n🔍 Matching Rules (Relaxed Matching):\n")

for i, row in fp_rules.iterrows():
    antecedents = set(row['antecedents'])

    # Count overlap
    match_count = len(antecedents.intersection(sample_set))

    if match_count > 0:
        print(f"Partial Match: {antecedents} → {row['consequents']}")
        print(f"Matched Features: {match_count}")
        print(f"Confidence: {row['confidence']:.2f}, Lift: {row['lift']:.2f}")
        print("------")

if not found:
    print("No strong rule found for this input")


🔹 APRIORI (Filtered Rules):

                        antecedents                consequents  support  \
53  (anxiety_level_Medium, grade_D)  (attendance_level_Medium)    0.129   
18                        (grade_B)        (anxiety_level_Low)    0.114   
8                         (grade_A)       (study_level_Medium)    0.100   
50    (sleep_level_Normal, grade_D)  (attendance_level_Medium)    0.129   
38    (study_level_Medium, grade_D)  (attendance_level_Medium)    0.172   

    confidence      lift  
53    0.895833  1.365600  
18    0.626374  1.278314  
8     0.943396  1.262913  
50    0.826923  1.260553  
38    0.826923  1.260553  

🔹 FP-GROWTH (Filtered Rules):

                        antecedents                consequents  support  \
14  (anxiety_level_Medium, grade_D)  (attendance_level_Medium)    0.129   
60                        (grade_B)        (anxiety_level_Low)    0.114   
27                        (grade_A)       (study_level_Medium)    0.100   
8     (sleep_level_Normal